# Excluding Time Segments from the GLM Fit with Masks

Recordings often contain segments that should not influence the model fit: motion
artifacts, periods in which an optode lost contact, or samples that are missing
altogether. Removing these samples from the time series is not an option, because the
GLM relies on the temporal structure of the data: the design matrix is defined on the
full time axis and the noise models account for temporal correlations.

Instead, `glm.fit` accepts a boolean `mask` that marks which samples enter the fit.
It follows the convention of the masks in `cedalion.sigproc.quality`:

- `True` (`quality.CLEAN`): the sample is used,
- `False` (`quality.TAINTED`): the sample is ignored.

The fit is determined by the unmasked samples only. The design matrix is not masked,
so `glm.predict` still provides a model prediction in the masked segments.

This notebook demonstrates masking first on simulated data with a known ground truth
and then on a real recording, in which motion artifacts are detected during
preprocessing and excluded from the fit.

In [ ]:
# This cells setups the environment when executed in Google Colab.
try:
    import google.colab
    !curl -s https://raw.githubusercontent.com/ibs-lab/cedalion/dev/scripts/colab_setup.py -o colab_setup.py
    # Select branch with --branch "branch name" (default is "dev")
    %run colab_setup.py
except ImportError:
    pass

In [ ]:
import matplotlib.pyplot as p
import numpy as np
import pandas as pd
import xarray as xr

import cedalion
import cedalion.data
import cedalion.dataclasses as cdc
import cedalion.models.glm as glm
import cedalion.nirs
import cedalion.sigproc.quality as quality
import cedalion.vis.blocks as vbx
import cedalion.vis.colors
import cedalion.xrutils as xrutils
from cedalion import units

xr.set_options(display_expand_data=False);

## Simulated data

### Time series with a known ground truth

We simulate two channels with the same hemodynamic response to five trials of
'StimA' and a constant offset. The HRF regressor is normalized to a peak value of 1,
so that the betas are the response amplitudes in µM.

In [ ]:
rng = np.random.default_rng(42)

fs = 10.0 * units.Hz  # sampling rate
T = 240 * units.s  # time series length
nsample = int(T * fs)

ts = cdc.build_timeseries(
    rng.normal(0, 0.1, (nsample, 2, 2)),
    dims=["time", "channel", "chromo"],
    time=np.arange(nsample) / fs.magnitude,
    channel=["S1D1", "S1D2"],
    value_units=units.uM,
    time_units=units.s,
    other_coords={"chromo": ["HbO", "HbR"]},
)

stim = pd.DataFrame(
    {
        "onset": [20.0, 60.0, 100.0, 140.0, 180.0],
        "trial_type": "StimA",
        "value": 1.0,
        "duration": 10.0,
    }
)

dms = glm.design_matrix.hrf_regressors(
    ts, stim, glm.Gamma(tau=0 * units.s, sigma=5 * units.s)
) & glm.design_matrix.drift_regressors(ts, drift_order=0)

# normalize the HRF regressor to a peak value of 1
hrf = dms.common.sel(regressor="HRF StimA")
dms.common.loc[{"regressor": "HRF StimA"}] = hrf / hrf.max("time")

# ground truth: response amplitude and offset in µM
true_betas = xr.DataArray(
    [[1.0, 0.5], [-0.3, 0.2]],
    dims=["chromo", "regressor"],
    coords={"chromo": ["HbO", "HbR"], "regressor": ["HRF StimA", "Drift 0"]},
).expand_dims(channel=ts.channel.values)

ts += xr.dot(dms.common, true_betas, dim="regressor").pint.quantify("uM")

### Corrupting the time series

We add two problems that frequently occur in real recordings:

- Between 100 s and 115 s, channel S1D1 contains a large artifact in both
  chromophores, e.g. from a movement of the probe. This segment overlaps with the
  third trial.
- Between 170 s and 175 s, the HbR signal of channel S1D2 is missing (NaN).

The mask marks both segments as `TAINTED`. `xrutils.mask` creates a mask with the
shape of the time series, in which all samples are `CLEAN`. Note that the mask of
S1D2 only excludes HbR, while the HbO samples of that segment remain in the fit.

In [ ]:
artifact = ((ts.time >= 100) & (ts.time < 115)).values
dropout = ((ts.time >= 170) & (ts.time < 175)).values

bump = np.sin(np.pi * (ts.time.values[artifact] - 100) / 15)
ts.loc[{"channel": "S1D1", "chromo": "HbO", "time": artifact}] += 3 * bump * units.uM
ts.loc[{"channel": "S1D1", "chromo": "HbR", "time": artifact}] += 1.5 * bump * units.uM
ts.loc[{"channel": "S1D2", "chromo": "HbR", "time": dropout}] = np.nan * units.uM

mask = xrutils.mask(ts, quality.CLEAN)
mask.loc[{"channel": "S1D1", "time": artifact}] = quality.TAINTED
mask.loc[{"channel": "S1D2", "chromo": "HbR", "time": dropout}] = quality.TAINTED

display(mask)

In [ ]:
# masked segments are shaded in gray
SEGMENT_FMT = {"ec": "none", "fc": "0.5", "alpha": 0.3}

f, ax = p.subplots(2, 1, sharex=True, figsize=(12, 5))
for i, ch in enumerate(ts.channel.values):
    ax[i].plot(ts.time, ts.sel(channel=ch, chromo="HbO"), "r-", lw=1, label="HbO")
    ax[i].plot(ts.time, ts.sel(channel=ch, chromo="HbR"), "b-", lw=1, label="HbR")
    vbx.plot_stim_markers(ax[i], stim)
    segments = quality.mask_to_segments(mask.sel(channel=ch).all("chromo"))
    vbx.plot_segments(ax[i], segments, fmt=SEGMENT_FMT, label="masked")
    ax[i].set_title(f"Ch. {ch}")
    ax[i].set_ylabel(r"$\Delta$ c / µM")
    ax[i].grid(True, alpha=0.3)
ax[0].legend(loc="upper left", ncol=3)
ax[1].set_xlabel("time / s")
p.tight_layout()

### Unmasked NaNs are an error

NaNs in the time series would invalidate the fit of the affected channel. Instead of
silently returning NaN results, `glm.fit` raises an exception if the time series
contains NaNs (or infs) in samples that are not masked. This catches errors in the
preprocessing early.

In [ ]:
try:
    glm.fit(ts, dms, noise_model="ols")
except ValueError as e:
    print(e)

### Fitting with a mask

The error message suggests `mask=np.isfinite(ts)`, which excludes exactly the
non-finite samples. This removes the NaNs, but the artifact in S1D1 remains in the
fit. We compare this fit with a fit that uses our mask, which excludes both segments.

In [ ]:
result_finite = glm.fit(ts, dms, noise_model="ols", mask=np.isfinite(ts))
result_masked = glm.fit(ts, dms, noise_model="ols", mask=mask)

pd.DataFrame(
    {
        "ground truth": true_betas.to_series(),
        "mask=np.isfinite(ts)": result_finite.sm.params.to_series(),
        "mask=mask": result_masked.sm.params.to_series(),
    }
).round(3)

The artifact biases the betas of S1D1 when it is not masked. With the mask, all betas
are close to the ground truth. The betas of S1D2 are the same in both fits, because
both masks exclude the same samples in this channel.

### Model predictions and residuals in masked segments

Because the design matrix is not masked, the model prediction is defined at all time
points. The fit with the mask predicts the response to the third trial, although the
data in this segment were ignored. The residuals (`.sm.resid`) are NaN in the masked
segment, since there is no data point that the model is compared to. They are labeled
with the sample index of the time series.

In [ ]:
pred_finite = glm.predict(ts, result_finite.sm.params, dms)
pred_masked = glm.predict(ts, result_masked.sm.params, dms)
resid = result_masked.sm.resid

ch = "S1D1"
segments = quality.mask_to_segments(mask.sel(channel=ch, chromo="HbO"))

f, ax = p.subplots(2, 1, sharex=True, figsize=(12, 6), height_ratios=[2, 1])
ax[0].plot(ts.time, ts.sel(channel=ch, chromo="HbO"), "-", c="0.6", lw=1, label="data")
ax[0].plot(
    pred_finite.time,
    pred_finite.sel(channel=ch, chromo="HbO"),
    "--",
    c="#eb6834",
    lw=2,
    label="model, mask=np.isfinite(ts)",
)
ax[0].plot(
    pred_masked.time,
    pred_masked.sel(channel=ch, chromo="HbO"),
    "-",
    c="#2a78d6",
    lw=2,
    label="model, mask=mask",
)
ax[0].set_ylabel(r"$\Delta$ HbO / µM")
ax[0].set_title(f"Ch. {ch}, HbO")

ax[1].plot(
    ts.time.values[resid.resid.values],
    resid.sel(channel=ch, chromo="HbO"),
    "-",
    c="0.3",
    lw=1,
)
ax[1].axhline(0, c="0.5", lw=1)
ax[1].set_ylabel("residual / µM")
ax[1].set_xlabel("time / s")
ax[1].set_title("residuals of the fit with mask=mask")

for a in ax:
    vbx.plot_segments(a, segments, fmt=SEGMENT_FMT, label="masked")
    a.grid(True, alpha=0.3)
ax[0].legend(loc="upper left", ncol=4)
ax[1].set_xlim(50, 170)
p.tight_layout()

### Mask shapes

The mask must have a time dimension. All other dimensions are optional and must be
dimensions of the time series. Missing dimensions are broadcast:

- a mask with dimension *time* masks a segment in all channels and chromophores,
- a mask with dimensions *time* and *channel* masks segments per channel, in all
  chromophores,
- a mask with dimensions *time*, *channel* and *chromo* masks segments per channel
  and chromophore.

For image reconstructions, *channel* is replaced by *parcel* or *vertex*. The plot
shows how each mask applies to the time series (red: `TAINTED`, blue: `CLEAN`).

In [ ]:
artifact_da = xr.DataArray(artifact, dims="time", coords={"time": ts.time})

masks = {
    "time": ~artifact_da,
    "time x channel": ~(artifact_da & (ts.channel == "S1D1")),
    "time x channel x chromo": ~(
        artifact_da & (ts.channel == "S1D1") & (ts.chromo == "HbO")
    ),
}

norm, cmap = cedalion.vis.colors.mask_cmap()
f, ax = p.subplots(1, 3, sharey=True, figsize=(12, 3.2))
for a, (name, m) in zip(ax, masks.items()):
    applied = (
        m.broadcast_like(mask)
        .stack(row=["channel", "chromo"])
        .transpose("row", "time")
    )
    im = a.pcolormesh(
        applied.time,
        np.arange(applied.sizes["row"]),
        applied,
        norm=norm,
        cmap=cmap,
        shading="nearest",
    )
    a.set_yticks(np.arange(applied.sizes["row"]))
    a.set_yticklabels([f"{c} {h}" for c, h in applied.row.values])
    a.set_title(f"mask dims:\n{name}")
    a.set_xlabel("time / s")
ax[0].invert_yaxis()
cb = f.colorbar(im, ax=ax, ticks=[0.25, 0.75])
cb.set_ticklabels(["TAINTED", "CLEAN"])

Masks can be combined with the usual boolean operators. Here, each mask is combined
with `np.isfinite(ts)` to also exclude the missing HbR samples of S1D2. Only the HRF
betas of S1D1 are shown. The artifact affects both chromophores of S1D1, so the mask
with the chromo dimension, which excludes only HbO, leaves the artifact in the HbR
fit.

In [ ]:
hrf_betas = {
    "ground truth": true_betas.sel(channel="S1D1", regressor="HRF StimA").to_series()
}
for name, m in masks.items():
    result = glm.fit(ts, dms, noise_model="ols", mask=m & np.isfinite(ts))
    hrf_betas[f"mask dims: {name}"] = result.sm.params.sel(
        channel="S1D1", regressor="HRF StimA"
    ).to_series()

pd.DataFrame(hrf_betas).round(3)

## Real data: excluding motion artifacts detected during preprocessing

### Loading the data and adding artifacts

We use the finger-tapping dataset and add short spikes to the optical density of ten
long-separation channels. Comparing the fits of the original data and the data with
spikes shows how much the spikes affect the betas and how well masking removes their
influence.

In [ ]:
rec = cedalion.data.get_fingertapping()

rec.stim.cd.rename_events(
    {
        "1.0": "control",
        "2.0": "Tapping/Left",
        "3.0": "Tapping/Right",
        "15.0": "sentinel",
    }
)
rec.stim = rec.stim[rec.stim.trial_type != "sentinel"]

rec["od"] = cedalion.nirs.cw.int2od(rec["amp"])

spiked_channels = [
    "S1D1", "S1D2", "S1D3", "S2D1", "S3D2", "S5D5", "S5D7", "S6D5", "S7D7", "S8D7"
]

rng = np.random.default_rng(1)
od_spikes = rec["od"].copy()
for ch in spiked_channels:
    for onset in rng.uniform(100, 2900, size=4):
        duration = rng.uniform(0.5, 2.0)
        amplitude = rng.choice([-1, 1]) * rng.uniform(0.2, 0.6)
        spike = ((od_spikes.time >= onset) & (od_spikes.time < onset + duration)).values
        od_spikes.loc[{"channel": ch, "time": spike}] += amplitude * units.dimensionless

rec["od_spikes"] = od_spikes

### Detecting motion artifacts

`quality.id_motion` marks segments with fast signal changes as `TAINTED`. It
operates on optical density and returns a mask with dimensions *channel*,
*wavelength* and *time*.

In [ ]:
ma_mask = quality.id_motion(
    rec["od_spikes"],
    t_motion=0.5 * units.s,
    t_mask=1.0 * units.s,
    stdev_thresh=7.0,
    amp_thresh=0.05,
)
display(ma_mask)

The GLM is fitted to concentrations, which have a *chromo* instead of a *wavelength*
dimension. Each concentration sample is calculated from both wavelengths, so it is
tainted if either wavelength is tainted. `quality.id_motion_refine` with operator
`"by_channel"` collapses the wavelength dimension accordingly. The operator `"all"`
would instead create a mask with only a time dimension that masks every artifact in
all channels.

In [ ]:
ma_mask_by_channel, ma_info = quality.id_motion_refine(ma_mask, "by_channel")
display(ma_mask_by_channel)
display(ma_info[ma_info.channel.isin(spiked_channels)])

In [ ]:
dpf = xr.DataArray(
    [6, 6],
    dims="wavelength",
    coords={"wavelength": rec["amp"].wavelength},
)
rec["conc"] = cedalion.nirs.cw.od2conc(rec["od"], rec.geo3d, dpf, spectrum="prahl")
rec["conc_spikes"] = cedalion.nirs.cw.od2conc(
    rec["od_spikes"], rec.geo3d, dpf, spectrum="prahl"
)

ts_long, ts_short = cedalion.nirs.split_long_short_channels(
    rec["conc"], rec.geo3d, distance_threshold=1.5 * units.cm
)
ts_long_spikes, _ = cedalion.nirs.split_long_short_channels(
    rec["conc_spikes"], rec.geo3d, distance_threshold=1.5 * units.cm
)

# The mask covers all channels. Select the long channels to match ts_long.
mask = ma_mask_by_channel.sel(channel=ts_long.channel)

In [ ]:
ch = "S5D5"
segments = quality.mask_to_segments(mask.sel(channel=ch))
t0 = segments[0][0]
window = ts_long_spikes.sel(channel=ch, time=slice(t0 - 30, t0 + 30))

f, ax = p.subplots(1, 1, figsize=(12, 4))
for chromo, color in [("HbO", "r"), ("HbR", "b")]:
    ax.plot(window.time, window.sel(chromo=chromo), "-", c=color, lw=1, label=chromo)
vbx.plot_stim_markers(ax, rec.stim)
vbx.plot_segments(ax, segments, fmt=SEGMENT_FMT, label="TAINTED (id_motion)")
ax.set_xlim(t0 - 30, t0 + 30)
ax.set_title(f"Ch. {ch} with spikes")
ax.set_xlabel("time / s")
ax.set_ylabel(r"$\Delta$ c / µM")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", ncol=3)
p.tight_layout()

### Fitting the GLM

The design matrix contains HRF regressors, polynomial drift regressors and the
closest short-separation channel. The spikes were added only to long channels, so
the same design matrix applies to the original data and the data with spikes.

In [ ]:
dms = (
    glm.design_matrix.hrf_regressors(
        ts_long,
        rec.stim,
        glm.Gamma(tau=0 * units.s, sigma=3 * units.s, T=3 * units.s),
    )
    & glm.design_matrix.drift_regressors(ts_long, drift_order=3)
    & glm.design_matrix.closest_short_channel_regressor(ts_long, ts_short, rec.geo3d)
)

The mask from `id_motion` has a wavelength dimension, which the concentration time
series lacks. Passing it to `glm.fit` raises an error that points to the solution.

In [ ]:
try:
    glm.fit(ts_long_spikes, dms, noise_model="ar_irls", mask=ma_mask)
except ValueError as e:
    print(e)

We fit the original data and the data with spikes, each with and without the motion
mask, using the AR-IRLS noise model.

In [ ]:
betas_orig = glm.fit(ts_long, dms, noise_model="ar_irls").sm.params
betas_spikes = glm.fit(ts_long_spikes, dms, noise_model="ar_irls").sm.params
betas_spikes_masked = glm.fit(
    ts_long_spikes, dms, noise_model="ar_irls", mask=mask
).sm.params
betas_orig_masked = glm.fit(
    ts_long, dms, noise_model="ar_irls", mask=mask
).sm.params

The plot compares the HRF betas of the spiked channels with the betas of the original
data. Without the mask, the spikes shift the betas away from the identity line. With
the mask, the betas are close to the original ones. The small remaining differences
arise because the fit with the mask uses fewer samples than the fit of the original
data.

In [ ]:
hrf_regressors = betas_orig.regressor.str.startswith("HRF ")
sel = {"channel": spiked_channels, "regressor": hrf_regressors}

f, ax = p.subplots(1, 2, sharex=True, sharey=True, figsize=(11, 5))
for a, (title, betas) in zip(
    ax,
    [("without mask", betas_spikes), ("with motion mask", betas_spikes_masked)],
):
    for chromo, color, marker in [("HbO", "r", "o"), ("HbR", "b", "s")]:
        a.scatter(
            betas_orig.sel(sel).sel(chromo=chromo).values.ravel(),
            betas.sel(sel).sel(chromo=chromo).values.ravel(),
            c=color,
            marker=marker,
            s=30,
            label=chromo,
        )
    a.set_title(f"data with spikes, {title}")
    a.set_xlabel(r"HRF $\beta$, original data")
    a.ticklabel_format(style="sci", scilimits=(0, 0))
    a.grid(True, alpha=0.3)
    a.set_aspect("equal")

lims = ax[0].get_xlim()
for a in ax:
    a.plot(lims, lims, "--", c="0.5", lw=1, label="identity", zorder=0)
ax[0].set_xlim(lims)
ax[0].set_ylim(lims)
ax[0].set_ylabel(r"HRF $\beta$, data with spikes")
ax[0].legend(loc="upper left")
p.tight_layout()

Fitting the original data with the same mask confirms that the masked samples have
no influence at all: the betas of the original data and of the data with spikes are
identical.

In [ ]:
max_diff = abs(betas_spikes_masked - betas_orig_masked).max().item()
print(f"max. difference of betas with the same mask: {max_diff}")

## Things to keep in mind

- **Masking excludes samples, it does not repair them.** It is suited for transient
  artifacts like spikes and for unusable segments. `id_motion` detects fast signal
  changes, so for a baseline shift only the jumps are marked, while the shifted
  segment stays in the fit. Baseline shifts should be corrected with a motion
  correction method before fitting.
- **Avoid filtering data with artifacts before the fit.** A temporal filter spreads
  an artifact into the neighboring samples, which are not masked. Model slow drifts
  with drift regressors in the design matrix instead.
- **AR-IRLS and gaps.** The prewhitening filter of order $p$ combines each sample
  with the $p$ preceding samples. After each masked segment, the first $p$ filtered
  samples are therefore excluded as well.
- **GLSAR** does not support masks with gaps between unmasked samples. Use AR-IRLS
  instead.
- **The design matrix must be finite.** `glm.fit` raises an exception if a regressor
  contains NaNs or infs.

## References

In [ ]:
cedalion.bib.dump_to_notebook()